# E1.1 — Gray Code vs Standard Binary Per Variable Type

Compares Gray-coded and standard binary encodings of the heterogeneous decision variables (host assignment, timing, well count) under DSBPSO's uniform bit-level stickiness.

**Experiment:** 5 encoding configurations × N seeds. Each configuration changes which variable types use Gray code vs standard binary.

| Config | Assignment | Timing | Wells |
|--------|-----------|--------|-------|
| all_gray | Gray | Gray | Gray |
| all_binary | Binary | Binary | Binary |
| assignment_binary | Binary | Gray | Gray |
| timing_binary | Gray | Binary | Gray |
| wells_binary | Gray | Gray | Binary |

**Metrics:** HVI (Hypervolume Indicator) and IGD (Inverted Generational Distance).

In [ ]:
import sys
sys.path.insert(0, "..")

import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from itertools import combinations
from scipy.stats import wilcoxon

def paired_wilcoxon(metric_a, seeds_a, metric_b, seeds_b):
    """Wilcoxon signed-rank paired by seed. Returns (p_value, n_common)."""
    seed_to_a = dict(zip(seeds_a, metric_a))
    seed_to_b = dict(zip(seeds_b, metric_b))
    common = sorted(set(seed_to_a) & set(seed_to_b))
    va = np.array([seed_to_a[s] for s in common])
    vb = np.array([seed_to_b[s] for s in common])
    if np.allclose(va, vb):
        return 1.0, len(common)
    return wilcoxon(va, vb, alternative="two-sided").pvalue, len(common)

from src.experiments.runner import load_experiment_results, load_run_result
from src.experiments.metrics import (
    hypervolume_2d,
    inverted_generational_distance,
    construct_reference_front,
)

# Shared config for all plots
config_order = ["all_gray", "all_binary", "assignment_binary", "timing_binary", "wells_binary"]
config_labels = ["All Gray", "All Binary", "Assign. Binary", "Timing Binary", "Wells Binary"]
colors = ["steelblue", "coral", "seagreen", "orchid", "goldenrod"]

## 1. Load Results

In [ ]:
runs = load_experiment_results("E1.1")
print(f"Loaded {len(runs)} runs")

# Group by config name
configs = defaultdict(list)
for run in runs:
    s = run["summary"]
    if s["status"] != "completed":
        print(f"  SKIPPING failed run: {s['run_id']}")
        continue
    configs[s["config_name"]].append(run)

for name, run_list in sorted(configs.items()):
    times = [r["summary"]["elapsed_seconds"] for r in run_list]
    print(f"  {name:20s}: {len(run_list)} runs, avg time {np.mean(times):.1f}s")

## 2. Extract Pareto Fronts and Compute Metrics

In [ ]:
# Extract re-evaluated Pareto front objectives from each run
fronts_by_config = {}
for name, run_list in configs.items():
    fronts = []
    for run in run_list:
        result = load_run_result(run["pickle_path"])
        pf = result["pareto_front"]
        if pf:
            obj = np.array([e.get("objectives_final", e["objectives"]) for e in pf])
            fronts.append(obj)
        else:
            fronts.append(np.empty((0, 2)))
    fronts_by_config[name] = fronts

print("Fronts extracted per config:")
for name, fronts in sorted(fronts_by_config.items()):
    sizes = [len(f) for f in fronts]
    print(f"  {name:20s}: {len(fronts)} runs, front sizes: {sizes}")

In [ ]:
# Construct reference front and reference point
all_objectives = [f for fronts in fronts_by_config.values() for f in fronts if len(f) > 0]
reference_front = construct_reference_front(all_objectives)
print(f"Reference front: {len(reference_front)} non-dominated solutions")

all_combined = np.vstack(all_objectives)
ref_point = np.array([
    all_combined[:, 0].min() - abs(all_combined[:, 0].min()) * 0.1,
    all_combined[:, 1].min() - abs(all_combined[:, 1].min()) * 0.1,
])
print(f"Reference point for HVI: {ref_point}")

In [ ]:
# Compute HVI and IGD per run
seeds_by_config = {name: [r["summary"]["seed"] for r in run_list] for name, run_list in configs.items()}
hvi_by_config = {}
igd_by_config = {}
for name, fronts in fronts_by_config.items():
    hvi_by_config[name] = np.array([hypervolume_2d(f, ref_point) for f in fronts])
    igd_by_config[name] = np.array([inverted_generational_distance(f, reference_front) for f in fronts])

print(f"{'Config':20s} {'HVI mean':>12s} {'HVI std':>10s} {'IGD mean':>12s} {'IGD std':>10s}")
print("-" * 66)
for name in config_order:
    if name not in hvi_by_config:
        continue
    hvi, igd = hvi_by_config[name], igd_by_config[name]
    print(f"{name:20s} {hvi.mean():12.1f} {hvi.std():10.1f} {igd.mean():12.4f} {igd.std():10.4f}")

## 3. Box Plots — HVI and IGD

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

active = [c for c in config_order if c in hvi_by_config]
labels = [config_labels[config_order.index(c)] for c in active]
hvi_data = [hvi_by_config[c] for c in active]
igd_data = [igd_by_config[c] for c in active]

bp1 = axes[0].boxplot(hvi_data, labels=labels, patch_artist=True)
for p in bp1["boxes"]: p.set_facecolor("steelblue"); p.set_alpha(0.7)
axes[0].set_ylabel("HVI"); axes[0].set_title("Hypervolume Indicator (higher = better)")
axes[0].tick_params(axis="x", rotation=30); axes[0].grid(True, alpha=0.3)

bp2 = axes[1].boxplot(igd_data, labels=labels, patch_artist=True)
for p in bp2["boxes"]: p.set_facecolor("seagreen"); p.set_alpha(0.7)
axes[1].set_ylabel("IGD"); axes[1].set_title("Inverted Generational Distance (lower = better)")
axes[1].tick_params(axis="x", rotation=30); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Overlaid Pareto Fronts (Representative Runs)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for i, name in enumerate(config_order):
    if name not in hvi_by_config: continue
    hvi_vals = hvi_by_config[name]
    median_idx = np.argmin(np.abs(hvi_vals - np.median(hvi_vals)))
    front = fronts_by_config[name][median_idx]
    if len(front) == 0: continue
    ax.scatter(front[:, 1], front[:, 0], color=colors[i], label=config_labels[i], s=30, alpha=0.8)

ax.set_xlabel("CVaR (MNOK)"); ax.set_ylabel("Expected ROV (MNOK)")
ax.set_title("Pareto Fronts — Median-HVI Run Per Encoding")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Statistical Significance — Pairwise Comparisons

Wilcoxon signed-rank tests (paired by seed) with Bonferroni correction.

In [ ]:
active_configs = [c for c in config_order if c in hvi_by_config]
n_configs = len(active_configs)
n_comparisons = n_configs * (n_configs - 1) // 2
bonferroni_alpha = 0.05 / n_comparisons

hvi_pvalues = np.ones((n_configs, n_configs))
igd_pvalues = np.ones((n_configs, n_configs))

for i, j in combinations(range(n_configs), 2):
    ci, cj = active_configs[i], active_configs[j]
    p, _ = paired_wilcoxon(hvi_by_config[ci], seeds_by_config[ci], hvi_by_config[cj], seeds_by_config[cj])
    hvi_pvalues[i, j] = hvi_pvalues[j, i] = p
    p, _ = paired_wilcoxon(igd_by_config[ci], seeds_by_config[ci], igd_by_config[cj], seeds_by_config[cj])
    igd_pvalues[i, j] = igd_pvalues[j, i] = p

print(f"Bonferroni-corrected alpha: {bonferroni_alpha:.4f} ({n_comparisons} comparisons)")

In [ ]:
active_labels = [config_labels[config_order.index(c)] for c in active_configs]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pvalues, title in [(axes[0], hvi_pvalues, "HVI p-values"), (axes[1], igd_pvalues, "IGD p-values")]:
    masked = pvalues.copy(); np.fill_diagonal(masked, np.nan)
    im = ax.imshow(masked, cmap="RdYlGn", vmin=0, vmax=0.1, aspect="equal")
    ax.set_xticks(range(n_configs)); ax.set_yticks(range(n_configs))
    ax.set_xticklabels(active_labels, rotation=45, ha="right")
    ax.set_yticklabels(active_labels); ax.set_title(title)
    for i in range(n_configs):
        for j in range(n_configs):
            if i == j: continue
            p = pvalues[i, j]
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < bonferroni_alpha else ""
            ax.text(j, i, f"{p:.3f}\n{sig}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout(); plt.show()
print(f"Significance: * p < {bonferroni_alpha:.4f} (Bonferroni), ** p < 0.01, *** p < 0.001")

## 6. Convergence Curves

Best E[ROV] and CVaR over iterations, averaged across seeds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, name in enumerate(config_order):
    if name not in configs: continue
    all_rov, all_cvar = [], []
    for run in configs[name]:
        hp = run["run_dir"] / "history.pkl"
        if hp.exists():
            with open(hp, "rb") as f: h = pickle.load(f)
            all_rov.append(h["best_rov"]); all_cvar.append(h["best_cvar"])
    if not all_rov: continue

    max_len = max(len(r) for r in all_rov)
    rov_arr = np.full((len(all_rov), max_len), np.nan)
    cvar_arr = np.full((len(all_cvar), max_len), np.nan)
    for j, (r, c) in enumerate(zip(all_rov, all_cvar)):
        rov_arr[j, :len(r)] = r; cvar_arr[j, :len(c)] = c

    iters = np.arange(1, max_len + 1)
    axes[0].plot(iters, np.nanmean(rov_arr, axis=0), color=colors[i], label=config_labels[i], alpha=0.8)
    axes[1].plot(iters, np.nanmean(cvar_arr, axis=0), color=colors[i], label=config_labels[i], alpha=0.8)

axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Best E[ROV] (MNOK)")
axes[0].set_title("Convergence — Best Expected ROV"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("Best CVaR (MNOK)")
axes[1].set_title("Convergence — Best CVaR"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Summary and Configuration Selection

Identifies the best encoding configuration based on HVI and IGD. This configuration is carried forward to E2.1.

In [ ]:
print("Ranking by mean HVI (higher = better):")
hvi_ranking = sorted([(n, hvi_by_config[n].mean()) for n in active_configs], key=lambda x: -x[1])
for rank, (name, val) in enumerate(hvi_ranking, 1):
    print(f"  {rank}. {config_labels[config_order.index(name)]:20s} HVI = {val:.1f}")

print("\nRanking by mean IGD (lower = better):")
igd_ranking = sorted([(n, igd_by_config[n].mean()) for n in active_configs], key=lambda x: x[1])
for rank, (name, val) in enumerate(igd_ranking, 1):
    print(f"  {rank}. {config_labels[config_order.index(name)]:20s} IGD = {val:.4f}")

best = hvi_ranking[0][0]
print(f"\nSelected encoding for E2.1: {config_labels[config_order.index(best)]}")
print(f"  Config key: {best}")